# 2. Post-Hoc Discretization Pipeline: The Latent Oddity Approach

This notebook implements a sophisticated two-stage VAE pipeline designed to overcome the limitations of standard Vector Quantization (VQ). By separating representation learning from discretization, we ensure a high-capacity latent space that is later organized based on the **decoder's geometric curvature**.

 1. **Continuous VAE Training**.
 We first train a Continuous Transformer VAE. The goal is to minimize the Evidence Lower Bound (ELBO) without the "noise" or non-differentiability of a discrete codebook.
   
    **Objective**: Learn a smooth mapping $q(z|x)$ and $p(x|z)$.       
    **Benefit**: Avoids "Index Collapse" and allows the latent space to utilize its full dimensionality.
    
    **Geometry** : At this stage, the latent space is numerically Euclidean but functionally warped by the Transformer layers.

 2. **Stage 2: Post-Hoc Riemannian Quantization**.
 Once the continuous space is stable, we apply the Latent Oddity discretization. We freeze the VAE and train an RBF-based Codebook using a non-Euclidean similarity metric.

 **The "Oddity" Metric Tensor $G(z)$**.

 Instead of clustering based on simple distance, we use the **Stochastic Riemannian Metric**:

 1. **Distortion $(G_{u}$)**: Measures how much the generated text Changes when moving in z. We use the **Jacobian of the Decoder** to identify where the space is "stretched".

 2. **Uncertainity $(G_{σ})$**: Measures the gradient of the VAE's predicted variance. The codebook is pushed away from "blurry" or noisy regions, ensuring embeddings settle in regions with high data density and low uncertainty.

## 2.1 Environment Setup and Repository Cloning

To ensure reproducibility, this section automates the setup of the working environment:
1. **Google Drive Integration:** Mounts your personal Drive to store persistent data (checkpoints and processed datasets).
2. **Project Structure:** Automatically creates a `DLAI` folder in your Drive.
3. **Dependency Management:** Installs the `uv` package manager and resolves all requirements defined in `pyproject.toml`.
4. **Source Code:** Clones the `llama` branch from our GitHub repository to provide access to the `src` module and configuration files.

**Note for Evaluators:** Please authorize the Google Drive mount when prompted to allow the notebook to save and retrieve project files.

In [ ]:
import os, sys

# 1. Mount Google Drive
# Evaluators will need to accept the pop-up to connect their Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Setup directories on Drive
# Create the DLAI folder if it doesn't exist on their Drive
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/DLAI"
if not os.path.exists(DRIVE_PROJECT_PATH):
    os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
    print(f"Created project folder at: {DRIVE_PROJECT_PATH}")

# 3. UV Installation
# We use UV for much faster dependency management than standard pip
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ['PATH']

# 4. Clone the Repository (Branch: llama)
# If the local folder doesn't exist, clone the specific branch
%cd /content
if not os.path.exists("DLAI"):
    !git clone --branch llama https://github.com/irene-30/DLAI.git
else:
    print("Repo already exists, pulling latest changes...")
    !git -C DLAI pull

# 5. Synchronize pyproject.toml
# Copy the pyproject.toml from the cloned repo to the Drive folder (if necessary)
# or vice versa, to ensure that UV reads the correct dependencies.
!cp /content/DLAI/pyproject.toml {DRIVE_PROJECT_PATH}/pyproject.toml

# 6. Install dependencies via pyproject.toml
# This command reads the .toml file and installs everything necessary
%cd /content/DLAI
!uv pip install -e . --system

# 7. Add to the system path to allow imports from 'src'
sys.path.append("/content/DLAI")
%cd /content

print("✅ Setup completed successfully!")

In [ ]:
# 1. Define the BASE folder on your Drive for experiments
DRIVE_BASE_FOLDER = "/content/drive/MyDrive/DLAI/experiments/oddity"

# 2. Define the exact file paths for model checkpoints
PATH_CONTINUOUS_VAE = os.path.join(DRIVE_BASE_FOLDER, "vae_continuous_uv.pth")
INTRA_EPOCH_CHECKPOINT = os.path.join(DRIVE_BASE_FOLDER, "vae_continuous_resume.pth")
BEST_MODEL_PATH = os.path.join(DRIVE_BASE_FOLDER, "vae_continuous_best.pth")

# 3. Create only the experiments folder, not the .pth folders
os.makedirs(DRIVE_BASE_FOLDER, exist_ok=True)

print(f"Main Model Path: {PATH_CONTINUOUS_VAE}")
print(f"Resume Path: {INTRA_EPOCH_CHECKPOINT}")

## Step 1: Continuous VAE Training

In this phase, we learn the smooth continuous manifold of the data. As defined in the generative model literature, we learn a deterministic surface mapping from the latent space $\mathcal{Z}$ to the input space $\mathcal{X}$. A continuous VAE serves as our foundation before we impose any discrete metric over it. We use an uncertainty-weighted Cross Entropy loss to approximate the local geometry behavior outlined in the Oddity formulation.

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from datasets import load_dataset
from tqdm import tqdm
from torch.amp import autocast, GradScaler
from typing import Tuple

from src.utils import get_llm_tokenizer, MAX_SEQ_LEN, VQ_CODEBOOK_SIZE
from src.dataset import Lazy_VQVAE_Dataset
from src.model.vae_oddity import ContinuousVAE

PER_DEVICE_BATCH_SIZE =  2
ACCUM_STEPS = 8
SAVE_FREQUENCY_BATCHES = 5000

# --- 1. Helper for Directory Creation ---
def save_checkpoint_with_dir_check(data, path):
    directory = os.path.dirname(path)
    if directory and not os.path.exists(directory):
        os.makedirs(directory, exist_ok=True)
    torch.save(data, path)

# --- 2. Optimized Validation Function ---
@torch.no_grad()
def validate_vae(model, val_loader, device, tokenizer) -> Tuple[float, float]:
    """
    Validation for Latent Oddity VAE.
    Calculates Reconstruction and KL losses manually using the Oddity formulation.
    """
    model.eval()
    total_val_recon_loss = 0
    total_val_kl_loss = 0

    max_val_steps = 500
    actual_steps = min(len(val_loader), max_val_steps)

    val_iter = iter(val_loader)

    # Use reduction='none' to weight each token with its predicted uncertainty
    loss_fn = torch.nn.CrossEntropyLoss(reduction='none', ignore_index=tokenizer.pad_token_id)

    for _ in range(actual_steps):
        try:
            batch = next(val_iter)
            input_ids = batch['input_ids'].to(device)

            # Use autocast for memory-efficient inference
            with autocast(device_type='cuda'):
                # 1. Forward Pass
                logits, logvar_dec, mu, logvar = model(input_ids)

                # 2. Manual Reconstruction Loss (Latent Space Oddity)
                ce_loss = loss_fn(logits.view(-1, logits.size(-1)), input_ids.view(-1))
                ce_loss = ce_loss.view(input_ids.shape)

                # Alignment and clamping of uncertainty
                if logvar_dec.dim() > 2:
                    logvar_dec = logvar_dec.mean(dim=-1)

                # Prevent numerical explosion during exponential application
                logvar_dec = torch.clamp(logvar_dec, min=-10.0, max=10.0)

                # Oddity Formula: Weight the raw error by the predicted variance
                weighted_recon = torch.exp(-logvar_dec) * ce_loss + 0.5 * logvar_dec
                recon = weighted_recon.mean()

                # 3. Manual KL Divergence Loss
                kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
                kl = kl / input_ids.size(0) # Normalize by batch size

            total_val_recon_loss += recon.item()
            total_val_kl_loss += kl.item()

            # VRAM cleanup for validation loop
            del logits, logvar_dec, mu, logvar, ce_loss, weighted_recon

        except StopIteration:
            break

    # Calculate average losses over the validation subset
    avg_recon = total_val_recon_loss / actual_steps
    avg_kl = total_val_kl_loss / actual_steps

    return avg_recon, avg_kl

In [ ]:
# --- 3. Data Loading ---
tokenizer = get_llm_tokenizer()

# Load a subset of MetaMathQA for training
print("Loading and tokenizing dataset...")
raw_dataset = load_dataset("meta-math/MetaMathQA")['train']
subset_dataset = raw_dataset.shuffle(seed=42).select(range(50000))

# Create Train/Test splits
split_datasets = subset_dataset.train_test_split(test_size=0.1, seed=42)

# Instantiate Lazy VQVAE Datasets
train_dataset = Lazy_VQVAE_Dataset(tokenizer, split_datasets['train'], max_length=MAX_SEQ_LEN)
val_dataset = Lazy_VQVAE_Dataset(tokenizer, split_datasets['test'], max_length=MAX_SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=PER_DEVICE_BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=PER_DEVICE_BATCH_SIZE, shuffle=False, num_workers=2)


In [ ]:
import torch.nn as nn

# --- 4. Main Training Function ---
def train_continuous_vae(num_epochs=1, lr=1e-4):
    print("--- 🚀 Step 1: Training Continuous VAE (Oddity Optimized) ---")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Initialize the model
    model = ContinuousVAE(
        vocab_size=len(tokenizer),
        d_model=256,
        max_seq_len=MAX_SEQ_LEN
    ).to(device)

    try:
        model.transformer_encoder.enable_gradient_checkpointing()
        model.transformer_decoder.enable_gradient_checkpointing()
        print("Enabled Gradient Checkpointing.")
    except:
        pass

    optimizer = optim.Adam(model.parameters(), lr=lr)
    BEST_VAL_RECON_LOSS = float('inf')
    scaler = GradScaler('cuda')

    start_epoch = 0
    start_batch_index = 0

    # Resume from checkpoint if it exists
    if os.path.exists(INTRA_EPOCH_CHECKPOINT) and os.path.isfile(INTRA_EPOCH_CHECKPOINT):
        print(f"Loading resume checkpoint: {INTRA_EPOCH_CHECKPOINT}")
        checkpoint = torch.load(INTRA_EPOCH_CHECKPOINT, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch']
        start_batch_index = checkpoint.get('batch_index', 0)
        BEST_VAL_RECON_LOSS = checkpoint.get('best_val_recon_loss', float('inf'))
        print(f"✅ SUCCESS: Resuming from Epoch {start_epoch}, Batch {start_batch_index}")
    else:
        print("ℹ️ Starting training from scratch (no valid resume file).")

    # Training Loop
    if start_epoch < num_epochs:
        for epoch in range(start_epoch, num_epochs):
            model.train()
            total_recon, total_kl = 0, 0

            # Fast skipping using Subset to resume intra-epoch
            if epoch == start_epoch and start_batch_index > 0:
                indices = list(range(start_batch_index * PER_DEVICE_BATCH_SIZE, len(train_dataset)))
                resumed_subset = Subset(train_dataset, indices)
                current_loader = DataLoader(resumed_subset, batch_size=PER_DEVICE_BATCH_SIZE, shuffle=False, num_workers=2)
                print(f"Fast-resumed from sample index {start_batch_index * PER_DEVICE_BATCH_SIZE}")
            else:
                current_loader = train_loader

            progress = tqdm(current_loader, initial=start_batch_index, total=len(train_loader), desc=f"Epoch {epoch+1}")

            optimizer.zero_grad()

            try:
                model.transformer_encoder.enable_gradient_checkpointing()
                model.transformer_decoder.enable_gradient_checkpointing()
            except AttributeError as e:
                pass

            # Define CrossEntropy with reduction='none' for the Latent Oddity formula
            loss_fn = nn.CrossEntropyLoss(reduction='none')

            for i_off, batch in enumerate(progress):
                i = i_off + (start_batch_index if epoch == start_epoch else 0)
                input_ids = batch['input_ids'].to(device)

                with autocast(device_type='cuda'):
                    logits, logvar_dec, mu, logvar = model(input_ids)

                    # --- Latent Space Oddity Loss ---
                    # 1. Calculate raw error for each token [Shape: Batch*SeqLen]]
                    ce_loss = loss_fn(logits.view(-1, logits.size(-1)), input_ids.view(-1))

                    # 2. Reshape [Batch*SeqLen] -> [Batch, SeqLen]
                    ce_loss = ce_loss.view(input_ids.shape)

                    # 3. Align uncertainty to [Batch, SeqLen] format
                    if logvar_dec.dim() > 2:
                        logvar_dec = logvar_dec.mean(dim=-1)

                    # Prevent numerical explosions by clamping logvar_dec
                    logvar_dec = torch.clamp(logvar_dec, min=-10.0, max=10.0)

                    # 4. Apply the Riemannian Equation weighting
                    weighted_recon = torch.exp(-logvar_dec) * ce_loss + 0.5 * logvar_dec

                    # 5. Final mean for the batch
                    recon_loss = weighted_recon.mean()

                    # Standard KL Divergence for the VAE
                    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / input_ids.size(0)

                    # Total loss formulation
                    loss = recon_loss + (0.0001 * kl_loss)
                    loss_normalized = loss / ACCUM_STEPS

                # Backward pass via scaler
                scaler.scale(loss_normalized).backward()

                # Step optimizer based on accumulation steps
                if (i + 1) % ACCUM_STEPS == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

                total_recon += recon_loss.item()
                total_kl += kl_loss.item()

                # Delete all heavy tensors to free up VRAM immediately
                del logits, logvar_dec, mu, logvar, ce_loss, weighted_recon, loss, loss_normalized

                if i % 10 == 0:
                    torch.cuda.empty_cache()

                # Intermediate Save
                if (i + 1) % SAVE_FREQUENCY_BATCHES == 0:
                    data_to_save = {
                        'epoch': epoch,
                        'batch_index': i + 1,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'best_val_recon_loss': BEST_VAL_RECON_LOSS,
                    }
                    save_checkpoint_with_dir_check(data_to_save, INTRA_EPOCH_CHECKPOINT)

                progress.set_description(f"R: {recon_loss.item():.4f} | K: {kl_loss.item():.2f}")

            # --- END OF EPOCH FORCED SAVE ---
            print(f"\nTraining Epoch {epoch+1} done. Saving transition checkpoint...")
            transition_checkpoint = {
                'epoch': epoch + 1,
                'batch_index': 0,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_recon_loss': BEST_VAL_RECON_LOSS,
            }
            save_checkpoint_with_dir_check(transition_checkpoint, INTRA_EPOCH_CHECKPOINT)

            # 4. Validation
            print(f"Validating Epoch {epoch+1}...")
            avg_val_recon, avg_val_kl = validate_vae(model, val_loader, device, tokenizer)
            print(f"Epoch {epoch+1} Summary: Val Recon: {avg_val_recon:.4f} | Val KL: {avg_val_kl:.4f}")

            if avg_val_recon < BEST_VAL_RECON_LOSS:
                BEST_VAL_RECON_LOSS = avg_val_recon
                # Save as best model
                save_checkpoint_with_dir_check(transition_checkpoint, BEST_MODEL_PATH)
                print(f"--> NEW BEST: {BEST_MODEL_PATH}")

            start_batch_index = 0 # Reset for next epoch

    # Final cleanup save
    save_checkpoint_with_dir_check(model.state_dict(), PATH_CONTINUOUS_VAE)
    print(f"Training Complete. Final model saved.")

    return model

if __name__ == "__main__":
    trained_vae_model = train_continuous_vae()

## Step 1.5: RBF Variance Extrapolation
Standard deep neural networks struggle to provide meaningful variance estimates in regions far from the training data, leading to a distorted view of the latent geometry.

To solve this, we model the precision (inverse variance) using a **Radial Basis Function (RBF) Network**. This forces the variance to naturally increase (uncertainty goes up) as we move away from known data centroids:
$$\beta_\psi(z) = W v(z) + \zeta$$
This creates high-variance "walls" that constrain the shortest paths and random walks to remain along the dense data manifold.

## Step 2: Post-Hoc Riemannian Discretization
Here, we compute the Riemannian distortion. We ensure that our latent discrete codebook is mapped according to the true data geometry (weighted by the RBF-estimated uncertainty).

In [ ]:
def train_variance_rbf(model, loader, tokenizer, num_epochs=1, lr=1e-3, accum_steps=4):
    print("--- 🚀 Step 1.5: Training RBF Variance Network (Ultra-Optimized) ---")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. FREEZE EVERYTHING EXCEPT THE RBF
    for name, param in model.named_parameters():
        if "variance_head" not in name:
            param.requires_grad = False
        else:
            param.requires_grad = True

    optimizer = torch.optim.Adam(model.variance_head.parameters(), lr=lr)
    ce_loss_fn = torch.nn.CrossEntropyLoss(reduction='none', ignore_index=tokenizer.pad_token_id)

    # Ensure transformer dropout layers are off
    model.eval()
    model.variance_head.train() # Turn on only the RBF in train mode

    optimizer.zero_grad()

    for epoch in range(num_epochs):
        progress = tqdm(loader, desc=f"Training RBF Epoch {epoch+1}")
        for i, batch in enumerate(progress):
            input_ids = batch['input_ids'].to(device)

            # ZONE 1: NO GRADIENTS (Saves Gigabytes of VRAM)
            with torch.no_grad():
                mu, logvar = model.encode(input_ids)
                z = model.reparameterize(mu, logvar)

                # Calculate the decoder's mean
                logits, _ = model.decode_from_z(z, tgt_tokens=input_ids)

                # Calculate the reconstruction error
                ce_loss = ce_loss_fn(logits.view(-1, logits.size(-1)), input_ids.view(-1))
                ce_loss = ce_loss.detach() # Detach it completely

                # Destroy logits to free memory
                del logits

            # ZONE 2: ACTIVE GRADIENTS ONLY FOR RBF
            # Pass 'z' to the RBF to compute uncertainty
            logvar_dec = model.variance_head(z)
            logvar_token = logvar_dec.mean(dim=-1).view(-1)

            # Calculate Negative Log-Likelihood (NLL)
            nll_loss = 0.5 * torch.exp(-logvar_token) * ce_loss + 0.5 * logvar_token

            mask = (input_ids.view(-1) != tokenizer.pad_token_id).float()
            loss = (nll_loss * mask).sum() / mask.sum()

            # Gradient Accumulation
            loss = loss / accum_steps
            loss.backward()

            if (i + 1) % accum_steps == 0:
                optimizer.step()
                optimizer.zero_grad()

            # Multiply by accum_steps just for correct printing
            progress.set_description(f"RBF NLL Loss: {loss.item() * accum_steps:.4f}")

    torch.save(model.state_dict(), PATH_CONTINUOUS_VAE)
    print("✅ RBF training completed and weights saved!")

train_variance_rbf(model=trained_vae_model, loader=train_loader, tokenizer=tokenizer)

In [ ]:
"""
Step 2 Script: Post-Hoc Discretization (Latent Oddity Edition) with Intra-Epoch Checkpoints.
1. Load trained Continuous VAE.
2. Freeze it.
3. Pass data through it to get 'mu'.
4. Train LatentOddityQuantizer using the Riemannian Metric.
"""

from src.model.latent_oddity import LatentOddityQuantizer

# --- PATHS AND PARAMETERS ---
PATH_ODDITY_QUANTIZER = "/content/drive/MyDrive/DLAI/experiments/oddity/oddity_quantizer_posthoc_uv.pth"
INTRA_EPOCH_CHECKPOINT_QUANT = "/content/drive/MyDrive/DLAI/experiments/oddity/quantizer_intra_epoch.pth"

SAVE_FREQUENCY_BATCHES = 2500

# --- 1. Validation Function ---
@torch.no_grad()
def evaluate_quantization(vae, quantizer, val_loader, device):
    quantizer.eval()
    vae.eval()
    total_distortion = 0
    max_steps = 100

    for i, batch in enumerate(tqdm(val_loader, desc="Validating Quantizer")):
        if i >= max_steps: break
        input_ids = batch['input_ids'].to(device)

        mu, logvar = vae.encode(input_ids)
        dist = torch.cdist(mu, quantizer.embedding.weight)
        min_dist, _ = torch.min(dist, dim=-1)
        total_distortion += min_dist.mean().item()

    return total_distortion / max_steps

# --- 2. Main Training Function ---
def train_posthoc_quantizer(vae, num_epochs=1, batch_size=2):
    print("--- 🚀 Step 2: Post-Hoc Riemannian Quantization (with Resume) ---")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = get_llm_tokenizer()
    raw_dataset = load_dataset("meta-math/MetaMathQA")['train']
    split_datasets = raw_dataset.train_test_split(test_size=0.1, seed=42)

    train_dataset = Lazy_VQVAE_Dataset(tokenizer, split_datasets['train'], max_length=MAX_SEQ_LEN)
    val_dataset = Lazy_VQVAE_Dataset(tokenizer, split_datasets['test'], max_length=MAX_SEQ_LEN)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Freeze VAE parameters
    vae.eval()
    for param in vae.parameters():
        param.requires_grad = False

    # 2. Initialize Latent Oddity Quantizer
    quantizer = LatentOddityQuantizer(
        vae_model=vae,
        num_embeddings=VQ_CODEBOOK_SIZE,
        embedding_dim=256,
        decay=0.99
    ).to(device)

    start_epoch = 0
    start_batch_index = 0

    if os.path.exists(INTRA_EPOCH_CHECKPOINT_QUANT) and os.path.isfile(INTRA_EPOCH_CHECKPOINT_QUANT):
        print(f"🔄 Found intra-epoch checkpoint: {INTRA_EPOCH_CHECKPOINT_QUANT}")
        checkpoint = torch.load(INTRA_EPOCH_CHECKPOINT_QUANT, map_location=device)
        quantizer.load_state_dict(checkpoint['model_state_dict'])
        start_epoch = checkpoint['epoch']
        start_batch_index = checkpoint.get('batch_index', 0)
        print(f"✅ SUCCESS: Resuming from Epoch {start_epoch+1}, Batch {start_batch_index}")
    else:
        print("ℹ️ No checkpoint found. Starting from scratch.")

    # 3. Clustering Loop
    if start_epoch < num_epochs:
        for epoch in range(start_epoch, num_epochs):
            print(f"\nEpoch {epoch+1}/{num_epochs} - Computing Riemannian Geometry...")
            quantizer.train()

            # Fast skipping usando Subset
            if epoch == start_epoch and start_batch_index > 0:
                indices = list(range(start_batch_index * batch_size, len(train_dataset)))
                resumed_subset = Subset(train_dataset, indices)
                # Set shuffle=False for the resumed subset for consistency,
                # even if the original train loader had True
                current_loader = DataLoader(resumed_subset, batch_size=batch_size, shuffle=False, num_workers=2)
                print(f"⏭️ Fast-forward: resuming from index {start_batch_index * batch_size}")
            else:
                current_loader = train_loader

            progress = tqdm(current_loader, initial=start_batch_index, total=len(train_loader), desc=f"Epoch {epoch+1}")

            for i_off, batch in enumerate(progress):
                i = i_off + (start_batch_index if epoch == start_epoch else 0)
                input_ids = batch['input_ids'].to(device)

                with torch.no_grad():
                    mu, _ = vae.encode(input_ids)

                mu.requires_grad_(True)
                quantizer.update(mu, input_ids=input_ids)

                if i % 10 == 0:
                    torch.cuda.empty_cache()

                # --- INTRA-EPOCH SAVING ---
                if (i + 1) % SAVE_FREQUENCY_BATCHES == 0:
                    data_to_save = {
                        'epoch': epoch,
                        'batch_index': i + 1,
                        'model_state_dict': quantizer.state_dict()
                    }
                    save_checkpoint_with_dir_check(data_to_save, INTRA_EPOCH_CHECKPOINT_QUANT)

            # --- END OF EPOCH SAVING (Transition) ---
            print(f"\n✅ Training Epoch {epoch+1} completed. Saving transition...")
            transition_checkpoint = {
                'epoch': epoch + 1,
                'batch_index': 0,
                'model_state_dict': quantizer.state_dict()
            }
            save_checkpoint_with_dir_check(transition_checkpoint, INTRA_EPOCH_CHECKPOINT_QUANT)

            # Reset the index for subsequent epochs
            start_batch_index = 0

    # 4. Final Validation & Saving
    print("\nStarting Validation...")
    avg_val_distortion = evaluate_quantization(vae, quantizer, val_loader, device)
    print(f"\n--- Evaluation Summary ---")
    print(f"Average Validation Quantization Distortion: {avg_val_distortion:.6f}")

    # Salvataggio Finale Pulito
    save_checkpoint_with_dir_check(quantizer.state_dict(), PATH_ODDITY_QUANTIZER)
    print(f"✅ Final model saved in {PATH_ODDITY_QUANTIZER}")

if __name__ == "__main__":
    train_posthoc_quantizer(vae=trained_vae_model)